# Chunk size, hybrid search and reranking in RAG on a Kaggle T4

Question: how do chunk size, retrieval method and reranking change retrieval quality? Does better retrieval give answers that stay closer to the source text?

Setup: two Tesla T4 GPUs (16 GB each, compute 7.5, fp16 only). The corpus is a set of public NVIDIA articles. They are also the reading list for the NVIDIA NCA-AMAD exam. The notebook fetches the pages at run time from their public URLs. It does not store article text in this repository.

Terms used in this notebook:

- **Chunk**: a block of consecutive words from one page. The retriever searches chunks, not whole pages.
- **BM25**: a keyword ranking function. It scores a chunk higher when it contains the query words, and more so for rare words.
- **Dense retrieval**: an embedding model turns the query and each chunk into a vector. The retriever ranks chunks by cosine similarity. It can match meaning when the words differ.
- **Hybrid search**: run BM25 and dense retrieval, then merge the two rankings.
- **Reciprocal rank fusion (RRF)**: a merge rule. Each chunk gets the sum of `1 / (k + rank)` over the rankings. This notebook uses `k = 60`.
- **Cross-encoder reranker**: a model that reads the query and one chunk together and gives one relevance score. It is slower than an embedding model, so it only reorders a short list.
- **recall@k**: the share of questions where at least one correct chunk is in the top k results.
- **MRR@10**: mean reciprocal rank. For each question, take `1 / rank` of the first correct chunk in the top 10. Use 0 if there is none. Then average over all questions.
- **Groundedness**: every claim in an answer is supported by the context given to the model.

## Predict before running

Write down your guesses first. Then compare them with the results.

1. Which chunk size gives the best recall@5: small, medium or large?
2. Does BM25 beat dense retrieval on these technical articles? Does hybrid beat both?
3. How much does reranking add to recall@1? How much latency does it cost per query?
4. Does answering from retrieved context raise the grounded rate compared with no context? Does reranked context beat plain hybrid context?

## Method

- **Eval set without hand labels.** Synthetic question answering is a standard way to test retrieval. The notebook samples passages of about 150 words. An LLM writes one question for each passage. The source page and the passage position are the gold answer.
- **Chunking.** Each page is split by words into chunks of 128, 256 and 512 words, with 15 percent overlap.
- **Hit rule.** A retrieved chunk is a hit when it comes from the gold page and it overlaps the gold passage by at least 50 percent of the passage words.
- **Known limit.** The embedding model and the reranker read at most 512 tokens. A 512-word chunk is longer than that, so those models see only the start of it. This hurts the largest chunk size and is part of the result.
- **GPU plan.** The vLLM server runs on GPU 0 for the whole notebook. The embedding model and the reranker run in the notebook on GPU 1. The server stays up, so there is no second start. With only one GPU, the notebook stops the server during retrieval and starts it again for answer generation.

In [ ]:
# 1. Install the packages Kaggle does not ship
!pip install -q vllm sentence-transformers rank-bm25 2>&1 | tail -3
!python -c "import vllm; print('vllm', vllm.__version__)"
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 2. Helper functions (CPU only, no GPU needed)
import re
import numpy as np

def tokenize(text):
    return re.findall(r"\w+", text.lower())

def chunk_words(words, size, overlap=0.15):
    """Return (start, end) word spans of `size` words. Step = size * (1 - overlap)."""
    step = max(1, int(size * (1 - overlap)))
    spans, s = [], 0
    while True:
        e = min(s + size, len(words))
        spans.append((s, e))
        if e >= len(words):
            return spans
        s += step

def is_hit(chunk, gold, min_frac=0.5):
    """chunk and gold are (doc, start, end, ...). Hit = same doc and overlap >= min_frac of gold words."""
    if chunk[0] != gold[0]:
        return False
    overlap = max(0, min(chunk[2], gold[2]) - max(chunk[1], gold[1]))
    return overlap >= min_frac * (gold[2] - gold[1])

def top_ids(scores, n=100):
    return np.argsort(-np.asarray(scores, dtype=float), kind="stable")[:n].tolist()

def rrf(rankings, k=60, n=100):
    """Reciprocal rank fusion. Ranks start at 1. Ties break by chunk id."""
    score = {}
    for ranking in rankings:
        for rank, i in enumerate(ranking, 1):
            score[i] = score.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(score, key=lambda i: (-score[i], i))[:n]

def recall_at_k(hits, k):
    return float(any(hits[:k]))

def mrr_at_k(hits, k=10):
    for rank, h in enumerate(hits[:k], 1):
        if h:
            return 1.0 / rank
    return 0.0

def evaluate(rankings, chunks, golds):
    r1 = r5 = mrr = 0.0
    for ranking, gold in zip(rankings, golds):
        hits = [is_hit(chunks[i], gold) for i in ranking[:10]]
        r1 += recall_at_k(hits, 1); r5 += recall_at_k(hits, 5); mrr += mrr_at_k(hits, 10)
    n = len(golds)
    return {"recall@1": r1 / n, "recall@5": r5 / n, "MRR@10": mrr / n}

def sample_passages(doc_lengths, n, size=150, per_doc=2, seed=0):
    """Pick n (doc, start, end) passages. Starts sit on a grid of `size` words, so passages never overlap."""
    import random
    rng = random.Random(seed)
    pool = []
    for d, length in enumerate(doc_lengths):
        starts = list(range(0, length - size + 1, size))
        rng.shuffle(starts)
        pool += [(d, s, s + size) for s in starts[:per_doc]]
    rng.shuffle(pool)
    return pool[:n]

In [ ]:
# 3. GPU layout and vLLM server helpers
import os, subprocess, time, json, urllib.request, requests, concurrent.futures as cf

n_gpu = len([l for l in subprocess.check_output(["nvidia-smi", "-L"], text=True).splitlines() if l.strip()])
SINGLE_GPU = n_gpu < 2
os.environ["CUDA_VISIBLE_DEVICES"] = "0" if SINGLE_GPU else "1"  # the notebook uses GPU 1; set before torch loads
print(f"{n_gpu} GPU(s) found. Notebook uses GPU {os.environ['CUDA_VISIBLE_DEVICES']}. SINGLE_GPU = {SINGLE_GPU}")

MODEL, PORT = "Qwen/Qwen2.5-7B-Instruct-AWQ", 8000
LOG = "/kaggle/working/vllm_server.log"
srv = None

def start_server():
    global srv
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")  # the server always gets GPU 0
    srv = subprocess.Popen(["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "4096",
                            "--gpu-memory-utilization", "0.85", "--port", str(PORT)],
                           stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env)
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen(f"http://localhost:{PORT}/health"); break
        except Exception:
            if srv.poll() is not None:
                print(open(LOG).read()[-4000:]); raise SystemExit("server exited")
            if time.time() - t0 > 1500: raise SystemExit("server not ready after 25 min")
            time.sleep(3)
    print(f"server ready after {time.time()-t0:.0f} s")

def stop_server():
    global srv
    if srv and srv.poll() is None:
        srv.terminate(); srv.wait(timeout=60)
    srv = None
    print("server stopped")

def chat(prompt, max_tokens=256, json_mode=False):
    body = {"model": MODEL, "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens, "temperature": 0}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"http://localhost:{PORT}/v1/chat/completions", json=body, timeout=300)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

def pmap(fn, items, workers=16):
    with cf.ThreadPoolExecutor(workers) as ex:
        return list(ex.map(fn, items))

## Step 1. Fetch the corpus

The cell below holds only the list of URLs. It downloads each page, keeps the main article text and drops navigation, scripts and styles. It skips pages that fail or have too little text, and it prints how many pages loaded. The corpus goes to `/kaggle/working/corpus.jsonl` so a rerun does not fetch again.

In [ ]:
URLS = [
"https://blogs.nvidia.com/blog/ai-blueprint-fraud-detection/",
"https://blogs.nvidia.com/blog/ai-query-engines-agentic-ai/",
"https://blogs.nvidia.com/blog/ai-tokens-explained/",
"https://blogs.nvidia.com/blog/canada-all-in/",
"https://blogs.nvidia.com/blog/nim-agent-blueprints/",
"https://blogs.nvidia.com/blog/open-models-data-ai/",
"https://blogs.nvidia.com/blog/three-computers-robotics/",
"https://blogs.nvidia.com/blog/what-is-retrieval-augmented-generation/",
"https://developer.nvidia.com/blog/7-drop-in-replacements-to-instantly-speed-up-your-python-data-science-workflows/",
"https://developer.nvidia.com/blog/a-simple-guide-to-deploying-generative-ai-with-nvidia-nim/",
"https://developer.nvidia.com/blog/advancing-anomaly-detection-for-industry-applications-with-nvidia-nv-tesseract-ad/",
"https://developer.nvidia.com/blog/an-easy-introduction-to-llm-reasoning-ai-agents-and-test-time-scaling/",
"https://developer.nvidia.com/blog/announcing-nvidia-merlin-application-framework-for-deep-recommender-systems/",
"https://developer.nvidia.com/blog/build-efficient-ai-agents-through-model-distillation-with-nvidias-data-flywheel-blueprint/",
"https://developer.nvidia.com/blog/building-for-the-rising-complexity-of-agentic-systems-with-extreme-co-design/",
"https://developer.nvidia.com/blog/building-nvidia-nemotron-3-agents-for-reasoning-multimodal-rag-voice-and-safety/",
"https://developer.nvidia.com/blog/building-your-first-llm-agent-application/",
"https://developer.nvidia.com/blog/closing-the-sim2real-gap-with-nvidia-isaac-sim-and-nvidia-isaac-replicator/",
"https://developer.nvidia.com/blog/demystifying-ai-inference-deployments-for-trillion-parameter-large-language-models/",
"https://developer.nvidia.com/blog/develop-multilingual-and-cross-lingual-information-retrieval-systems-with-efficient-data-storage/",
"https://developer.nvidia.com/blog/enhance-your-ai-agent-with-data-flywheels-using-nvidia-nemo-microservices/",
"https://developer.nvidia.com/blog/enhancing-rag-applications-with-nvidia-nim/",
"https://developer.nvidia.com/blog/finding-the-best-chunking-strategy-for-accurate-ai-responses/",
"https://developer.nvidia.com/blog/fine-tune-and-align-llms-easily-with-nvidia-nemo-customizer/",
"https://developer.nvidia.com/blog/gpus-for-etl-optimizing-etl-architecture-for-apache-spark-sql-operations/",
"https://developer.nvidia.com/blog/gpus-for-etl-run-faster-less-costly-workloads-with-nvidia-rapids-accelerator-for-apache-spark-and-databricks/",
"https://developer.nvidia.com/blog/grandmaster-pro-tip-winning-first-place-in-kaggle-competition-with-feature-engineering-using-nvidia-cudf-pandas/",
"https://developer.nvidia.com/blog/horizontal-autoscaling-of-nvidia-nim-microservices-on-kubernetes/",
"https://developer.nvidia.com/blog/how-small-language-models-are-key-to-scalable-agentic-ai/",
"https://developer.nvidia.com/blog/inside-nvidia-blackwell-ultra-the-chip-powering-the-ai-factory-era/",
"https://developer.nvidia.com/blog/introducing-new-kv-cache-reuse-optimizations-in-nvidia-tensorrt-llm/",
"https://developer.nvidia.com/blog/introduction-to-llm-agents/",
"https://developer.nvidia.com/blog/llm-benchmarking-fundamental-concepts/",
"https://developer.nvidia.com/blog/mastering-agentic-techniques-ai-agent-customization/",
"https://developer.nvidia.com/blog/mastering-agentic-techniques-ai-agent-evaluation/",
"https://developer.nvidia.com/blog/mastering-llm-techniques-data-preprocessing/",
"https://developer.nvidia.com/blog/mastering-llm-techniques-evaluation/",
"https://developer.nvidia.com/blog/mastering-llm-techniques-inference-optimization/",
"https://developer.nvidia.com/blog/maximize-ai-agent-performance-with-data-flywheels-using-nvidia-nemo-microservices/",
"https://developer.nvidia.com/blog/maximizing-gpu-utilization-with-nvidia-runai-and-nvidia-nim/",
"https://developer.nvidia.com/blog/nvidia-nemo-retriever-delivers-accurate-multimodal-pdf-data-extraction-15x-faster/",
"https://developer.nvidia.com/blog/nvidia-nim-offers-optimized-inference-microservices-for-deploying-ai-models-at-scale/",
"https://developer.nvidia.com/blog/nvidia-tensorrt-llm-now-accelerates-encoder-decoder-models-with-in-flight-batching/",
"https://developer.nvidia.com/blog/optimizing-inference-efficiency-for-llms-at-scale-with-nvidia-nim-microservices/",
"https://developer.nvidia.com/blog/optimizing-llms-for-performance-and-accuracy-with-post-training-quantization/",
"https://developer.nvidia.com/blog/rag-101-demystifying-retrieval-augmented-generation-pipelines/",
"https://developer.nvidia.com/blog/rag-101-retrieval-augmented-generation-questions-answered/",
"https://developer.nvidia.com/blog/route-ai-agent-workloads-across-models-with-nvidia-nemo-switchyard/",
"https://developer.nvidia.com/blog/seamlessly-deploying-a-swarm-of-lora-adapters-with-nvidia-nim/",
"https://developer.nvidia.com/blog/selecting-large-language-model-customization-techniques/",
"https://developer.nvidia.com/blog/streamline-evaluation-of-llms-for-accuracy-with-nvidia-nemo-evaluator/",
"https://developer.nvidia.com/blog/the-kaggle-grandmasters-playbook-7-battle-tested-modeling-techniques-for-tabular-data/",
"https://developer.nvidia.com/blog/training-instance-segmentation-models-using-maskrcnn-on-tao-toolkit/",
"https://developer.nvidia.com/blog/tune-and-deploy-lora-llms-with-nvidia-tensorrt-llm/",
"https://developer.nvidia.com/blog/vision-language-model-prompt-engineering-guide-for-image-and-video-understanding/",
"https://docs.isaacsim.omniverse.nvidia.com/6.0.0/replicator_tutorials/index.html",
"https://docs.nvidia.com/ai-enterprise/planning-resource/ai-factory-reference-design-for-government-white-paper/latest/observability.html",
"https://docs.nvidia.com/ai-enterprise/planning-resource/ai-factory-white-paper/latest/ai-factory-overview.html",
"https://docs.nvidia.com/deeplearning/triton-inference-server/archives/triton-inference-server-2600/user-guide/docs/tensorrtllm_backend/docs/guided_decoding.html",
"https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html",
"https://docs.nvidia.com/learning/physical-ai/",
"https://docs.nvidia.com/ncx/ncp-software-reference-guide/latest/telemetry-observability.html",
"https://docs.nvidia.com/nemo/agent-toolkit/latest/build-workflows/advanced/interactive-workflows.html",
"https://docs.nvidia.com/nemo/agent-toolkit/latest/build-workflows/mcp-client.html",
"https://docs.nvidia.com/nemo/agent-toolkit/latest/improve-workflows/evaluate.html",
"https://docs.nvidia.com/nemo/guardrails/about-nemo-guardrails-library/rail-types",
"https://docs.nvidia.com/nemo/guardrails/configure-guardrails/guardrail-catalog/fact-checking",
"https://docs.nvidia.com/nemo/megatron-bridge/latest/parallelisms.html",
"https://docs.nvidia.com/nemo/microservices/26.3.1/evaluator/benchmarks/index.html",
"https://docs.nvidia.com/nim-operator/latest/air-gap.html",
"https://docs.nvidia.com/nim-operator/latest/cache.html",
"https://docs.nvidia.com/nim-operator/latest/index.html",
"https://docs.nvidia.com/nim/benchmarking/llm/latest/metrics.html",
"https://docs.nvidia.com/nim/benchmarking/llm/latest/overview.html",
"https://docs.nvidia.com/nim/benchmarking/llm/latest/parameters.html",
"https://docs.nvidia.com/nim/large-language-models/latest/api-reference.html",
"https://docs.nvidia.com/nim/large-language-models/latest/troubleshooting/memory.html",
"https://docs.nvidia.com/nim/nemo-retriever/text-reranking/latest/overview.html",
"https://docs.nvidia.com/nvcf/runbooks",
"https://docs.nvidia.com/rag/latest/enable-nemotron-thinking.html",
"https://docs.nvidia.com/rag/latest/hybrid_search.html",
"https://docs.nvidia.com/rag/latest/multiturn.html",
"https://docs.nvidia.com/rag/latest/prompt-customization.html",
"https://docs.nvidia.com/tao/tao-toolkit/latest/text/overview.html",
"https://learn.nvidia.com/courses/course-detail?course_id=course-v1:DLI+C-FX-30+V1",
"https://nvidianews.nvidia.com/news/dsx-infrastructure-ai-factory",
"https://www.nvidia.com/en-us/ai-data-science/foundation-models/nemotron/",
"https://www.nvidia.com/en-us/data-center/products/ai-enterprise/",
"https://www.nvidia.com/en-us/glossary/ai-agents/",
"https://www.nvidia.com/en-us/glossary/computer-vision/",
"https://www.nvidia.com/en-us/glossary/data-flywheel/",
"https://www.nvidia.com/en-us/glossary/deep-learning/",
"https://www.nvidia.com/en-us/glossary/generative-physical-ai/",
"https://www.nvidia.com/en-us/glossary/multi-agent-systems/",
"https://www.nvidia.com/en-us/glossary/synthetic-data-generation/",
"https://www.nvidia.com/en-us/glossary/vector-database/",
"https://www.nvidia.com/en-us/learn/certification/ai-model-application-development-associate/"
]

In [ ]:
# 4. Fetch pages and extract article text
from bs4 import BeautifulSoup

CORPUS_PATH = "/kaggle/working/corpus.jsonl"
MIN_WORDS = 200
HEADERS = {"User-Agent": "gpu-experiments-notebook"}

def fetch_page(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "header", "footer", "aside", "form", "noscript", "svg"]):
            tag.decompose()
        node = soup.find("article") or soup.find("main") or soup.body
        text = re.sub(r"\s+", " ", node.get_text(" ")).strip() if node else ""
        title = soup.title.get_text(strip=True) if soup.title else url
        return {"url": url, "title": title, "text": text}
    except Exception as e:
        print("skip", url, type(e).__name__)
        return None

if os.path.exists(CORPUS_PATH):
    corpus = [json.loads(l) for l in open(CORPUS_PATH)]
    print("loaded cached corpus")
else:
    pages = pmap(fetch_page, URLS, workers=6)  # low rate: be polite to the site
    corpus = [p for p in pages if p and len(p["text"].split()) >= MIN_WORDS]
    with open(CORPUS_PATH, "w") as f:
        for p in corpus:
            f.write(json.dumps(p) + "\n")
    print(f"{sum(p is not None for p in pages)} of {len(URLS)} pages fetched, {len(corpus)} kept (at least {MIN_WORDS} words)")

DOCW = [p["text"].split() for p in corpus]
print(f"{len(corpus)} pages, {sum(map(len, DOCW)):,} words in total, median {int(np.median([len(w) for w in DOCW]))} words per page")

## Step 2. Build the eval set

The notebook samples 120 passages of about 150 words with a fixed seed. The vLLM server hosts `Qwen/Qwen2.5-7B-Instruct-AWQ` on GPU 0. For each passage the model writes one question that only that passage can answer. The cell drops outputs that are not valid JSON and questions that point at "the passage" or "the text".

The server stays up after this step, because step 6 needs it again. The embedding model and the reranker use GPU 1, so there is no memory conflict. Restarting the server would cost several minutes for no gain.

In [ ]:
# 5. Start vLLM and generate one question per sampled passage
start_server()

PASSAGE_WORDS, N_PASSAGES = 150, 120
golds_all = sample_passages([len(w) for w in DOCW], N_PASSAGES, PASSAGE_WORDS, seed=0)

PROMPT_Q = ("Read the passage and write ONE specific question that can be answered only from this passage. "
            "Do not use words like 'the passage', 'the text' or 'the article'. "
            'Reply with JSON only: {{"question": "..."}}\n\nPassage:\n{passage}')
BAD = re.compile(r"\b(the|this) (passage|text|article|page)\b", re.I)

def make_question(gold):
    d, s, e = gold
    passage = " ".join(DOCW[d][s:e])
    out = parse_json(chat(PROMPT_Q.format(passage=passage), max_tokens=100, json_mode=True))
    q = out.get("question") if isinstance(out, dict) else None
    if not isinstance(q, str) or len(q.split()) < 5 or BAD.search(q):
        return None
    return {"q": q.strip(), "gold": gold, "passage": passage}

t0 = time.time()
evalset = [x for x in pmap(make_question, golds_all) if x]
print(f"{len(evalset)} of {len(golds_all)} questions kept in {time.time()-t0:.0f} s ({len(golds_all)-len(evalset)} dropped)")
for x in evalset[:3]:
    print("-", x["q"])
QS = [x["q"] for x in evalset]
GOLDS = [x["gold"] for x in evalset]

## Step 3. Chunk, retrieve and score

For each chunk size the notebook builds three rankings of the top 100 chunks per question:

- **BM25** from `rank_bm25`, with lower-cased word tokens.
- **Dense** from `BAAI/bge-small-en-v1.5`. Embeddings are normalized, so the dot product is the cosine similarity. Queries get the instruction prefix that the model card recommends.
- **Hybrid** from reciprocal rank fusion of the two rankings with `k = 60`.

Then it scores each ranking with the hit rule from the method section.

In [ ]:
# 6. Chunk, index, retrieve, score
import torch, pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

if SINGLE_GPU:
    stop_server()  # one GPU: free it for the embedding model

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
embedder.half(); embedder.max_seq_length = 512
QPREFIX = "Represent this sentence for searching relevant passages: "
qvec = embedder.encode([QPREFIX + q for q in QS], normalize_embeddings=True, convert_to_numpy=True)

SIZES = [128, 256, 512]
IDX, rows = {}, []
for size in SIZES:
    chunks = [(d, s, e, " ".join(w[s:e])) for d, w in enumerate(DOCW) for s, e in chunk_words(w, size)]
    bm25 = BM25Okapi([tokenize(c[3]) for c in chunks])
    cvec = embedder.encode([c[3] for c in chunks], batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    rk = {"BM25": [top_ids(bm25.get_scores(tokenize(q))) for q in QS],
          "Dense": [top_ids(cvec @ qv) for qv in qvec]}
    rk["Hybrid (RRF)"] = [rrf([a, b]) for a, b in zip(rk["BM25"], rk["Dense"])]
    IDX[size] = {"chunks": chunks, "rank": rk}
    for method, rankings in rk.items():
        rows.append({"chunk words": size, "method": method, "chunks": len(chunks),
                     **evaluate(rankings, chunks, GOLDS)})
    print(f"chunk size {size}: {len(chunks)} chunks indexed")

retrieval = pd.DataFrame(rows)
print(retrieval.round(3).to_string(index=False))

In [ ]:
# 7. Bar chart: one panel per metric
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
methods = ["BM25", "Dense", "Hybrid (RRF)"]
for ax, metric in zip(axes, ["recall@1", "recall@5", "MRR@10"]):
    for j, m in enumerate(methods):
        vals = [retrieval[(retrieval["chunk words"] == s) & (retrieval.method == m)][metric].iloc[0] for s in SIZES]
        ax.bar(np.arange(len(SIZES)) + (j - 1) * 0.27, vals, 0.27, label=m)
    ax.set_xticks(range(len(SIZES))); ax.set_xticklabels([f"{s} words" for s in SIZES])
    ax.set_title(metric); ax.set_ylim(0, 1)
axes[0].legend(); axes[0].set_ylabel("score (higher is better)")
plt.tight_layout(); plt.savefig("/kaggle/working/retrieval_by_chunk_size.png", dpi=130); plt.show()

## Step 4. Rerank the hybrid results

Pick the chunk size with the best hybrid recall@5 (ties go to MRR@10). Take the top 20 hybrid chunks for each question. A cross-encoder, `BAAI/bge-reranker-base`, scores each (question, chunk) pair. The notebook reorders the 20 chunks by that score. It also times the reranker per question. That time is the latency the reranker adds on top of retrieval.

In [ ]:
# 8. Rerank the hybrid top 20 with a cross-encoder
from sentence_transformers import CrossEncoder

hyb = retrieval[retrieval.method == "Hybrid (RRF)"].sort_values(["recall@5", "MRR@10"], ascending=False)
BEST = int(hyb.iloc[0]["chunk words"])
chunks, hybrid_rank = IDX[BEST]["chunks"], IDX[BEST]["rank"]["Hybrid (RRF)"]
print("best chunk size for hybrid:", BEST, "words")

reranker = CrossEncoder("BAAI/bge-reranker-base", device="cuda", max_length=512)
try: reranker.model.half()
except Exception as e: print("fp16 not applied:", e)
reranker.predict([(QS[0], chunks[hybrid_rank[0][0]][3])] * 20)  # warm up

reranked, lat_ms = [], []
for q, cand in zip(QS, hybrid_rank):
    top = cand[:20]
    t0 = time.perf_counter()
    scores = reranker.predict([(q, chunks[i][3]) for i in top], batch_size=20)
    lat_ms.append((time.perf_counter() - t0) * 1000)
    reranked.append([top[j] for j in top_ids(scores, 20)])

m_hyb = evaluate(hybrid_rank, chunks, GOLDS)
m_rr = evaluate(reranked, chunks, GOLDS)
rerank_df = pd.DataFrame([{"stage": "Hybrid (RRF)", **m_hyb, "added ms/query": 0.0},
                          {"stage": "Hybrid + rerank top 20", **m_rr, "added ms/query": float(np.mean(lat_ms))}])
print(rerank_df.round(3).to_string(index=False))
print(f"reranker latency: median {np.median(lat_ms):.0f} ms, p95 {np.percentile(lat_ms, 95):.0f} ms per query (20 chunks)")

## Step 5. Answer quality and groundedness

For the first 30 questions, the vLLM model answers in three ways:

- **(a) No context.** The model answers from its own knowledge.
- **(b) Top 5 hybrid chunks** as context.
- **(c) Top 5 reranked chunks** as context.

The same model then acts as a judge. The prompt asks whether every claim in the answer appears in the context, and the judge replies yes or no as JSON. For (b) and (c) the context is the one the answer used. For (a) there is no context, so the judge uses the hybrid top 5. That checks whether the unaided answer says things the retrieved text does not support.

To fit the 4096-token window, the notebook shortens each chunk so the five chunks together hold at most 1800 words.

An LLM judge is a noisy proxy. It can say yes for a loose paraphrase and no for a correct answer written in other words. The same model writes and judges the answers, which can bias the result. Use the grounded rate to compare conditions on this fixed set. Do not read it as a true hallucination rate.

In [ ]:
# 9. Generate answers in three conditions and judge groundedness
if srv is None or srv.poll() is not None:
    start_server()

N_ANS, CTX_WORDS = 30, 1800

def build_context(ids):
    per = CTX_WORDS // len(ids)
    return "\n\n".join(f"[{n+1}] " + " ".join(chunks[i][3].split()[:per]) for n, i in enumerate(ids))

PROMPT_RAG = "Answer the question using only the context. Be brief.\n\nContext:\n{ctx}\n\nQuestion: {q}\nAnswer:"
PROMPT_NOCTX = "Answer the question briefly.\n\nQuestion: {q}\nAnswer:"
PROMPT_JUDGE = ('Context:\n{ctx}\n\nAnswer:\n{ans}\n\nDoes every claim in the answer appear in the context? '
                'Reply with JSON only: {{"grounded": "yes"}} or {{"grounded": "no"}}')

ctx_b = [build_context(hybrid_rank[k][:5]) for k in range(N_ANS)]
ctx_c = [build_context(reranked[k][:5]) for k in range(N_ANS)]
conds = {
    "(a) no context":     (lambda k: PROMPT_NOCTX.format(q=QS[k]), ctx_b),   # judged against the hybrid context
    "(b) top-5 hybrid":   (lambda k: PROMPT_RAG.format(ctx=ctx_b[k], q=QS[k]), ctx_b),
    "(c) top-5 reranked": (lambda k: PROMPT_RAG.format(ctx=ctx_c[k], q=QS[k]), ctx_c),
}

def judge(args):
    ctx, ans = args
    out = parse_json(chat(PROMPT_JUDGE.format(ctx=ctx, ans=ans), max_tokens=20, json_mode=True))
    v = str(out.get("grounded", "")).lower() if isinstance(out, dict) else ""
    return {"yes": True, "no": False}.get(v)

answers, ans_rows = {}, []
for name, (make_prompt, judge_ctx) in conds.items():
    answers[name] = pmap(lambda k: chat(make_prompt(k), max_tokens=200), range(N_ANS))
    verdicts = pmap(judge, list(zip(judge_ctx, answers[name])))
    scored = [v for v in verdicts if v is not None]
    ans_rows.append({"condition": name, "answers": N_ANS, "judged": len(scored),
                     "grounded rate": sum(scored) / max(1, len(scored))})
answer_df = pd.DataFrame(ans_rows)
print(answer_df.round(3).to_string(index=False))
stop_server()

## Step 6. Summary

The table below joins all results. The notebook saves it to `/kaggle/working/rag_results.csv`.

In [ ]:
# 10. Final comparison table
final = []
for _, r in retrieval.iterrows():
    final.append({"experiment": "retrieval", "config": f"{int(r['chunk words'])} words, {r['method']}",
                  "recall@1": r["recall@1"], "recall@5": r["recall@5"], "MRR@10": r["MRR@10"],
                  "extra": f"{int(r['chunks'])} chunks"})
final.append({"experiment": "rerank", "config": f"{BEST} words, hybrid + rerank top 20",
              "recall@1": m_rr["recall@1"], "recall@5": m_rr["recall@5"], "MRR@10": m_rr["MRR@10"],
              "extra": f"+{np.mean(lat_ms):.0f} ms/query"})
for _, r in answer_df.iterrows():
    final.append({"experiment": "answers", "config": r["condition"], "recall@1": None, "recall@5": None,
                  "MRR@10": None, "extra": f"grounded {r['grounded rate']:.2f} ({int(r['judged'])} judged)"})
final_df = pd.DataFrame(final)
final_df.to_csv("/kaggle/working/rag_results.csv", index=False)
print(final_df.round(3).to_string(index=False))
print(f"eval set: {len(evalset)} questions over {len(corpus)} pages")

## What to look for

- Compare the three chunk sizes. Small chunks point at the right text but carry less context. Large chunks carry more context but dilute the match, and the embedding model cuts them at 512 tokens.
- Compare BM25 and dense. The synthetic questions often reuse words from the passage, which favors BM25. Check whether hybrid still helps.
- Compare recall@1 before and after reranking. Weigh the gain against the added milliseconds per query.
- Compare grounded rates. Then compare the answer step with the retrieval step. Better retrieval does not always change the answer.
- Limits: one corpus, one eval set made by one LLM, and a small answer sample. Treat small gaps as noise.

## Exam rules this proves

- **customization.7**: Retrieval quality depends on the ingestion choices. Chunk size and overlap change what the retriever can find, so measure them instead of using defaults.
- **customization.8**: Semantic (dense) search and keyword search fail on different questions. Hybrid search with rank fusion combines both rankings.
- **customization.9**: Putting retrieved text in the prompt is the cheapest way to add facts and to make answers checkable. Compare it with prompting alone before you consider fine-tuning.
- **models.4**: An embedding model finds candidates fast. A cross-encoder reranker orders a short list more carefully and costs more time per query.
- **data.7**: Test a RAG system on your own documents with questions made from them. A general benchmark does not show how your chunks and queries behave.
- **data.12**: Check whether each answer is supported by the retrieved context. An LLM judge gives a cheap but noisy groundedness signal.